# Resume Analyser — Runnables & Memory

This notebook builds a **multi-turn resume analyser** step by step, introducing:

| Concept | What it does |
|---|---|
| `RunnableLambda` | Wraps any Python function into a chain step |
| `RunnablePassthrough` | Passes input unchanged to the next step |
| `RunnableParallel` | Runs multiple chains at the same time on the same input |
| `RunnableBranch` | Routes input to different chains based on a condition |
| `InMemoryChatMessageHistory` | In-process store for conversation turns |
| `RunnableWithMessageHistory` | Adds per-session memory to any chain |
| `MemorySaver` | Adds persistent memory to a LangGraph agent |

---
## 1. Setup

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnableLambda,
    RunnablePassthrough,
    RunnableParallel,
    RunnableBranch,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, WebBaseLoader
import tiktoken

llm = ChatOpenAI(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
    max_tokens=2000,
)
parser = StrOutputParser()
print("Setup complete!")

---
## 2. Helper Utilities

Two plain functions before we turn them into Runnables:
- `truncate_text` — keeps input within the model's token budget
- `load_document` — reads PDF, DOCX, or a web URL and returns plain text

In [ ]:
def truncate_text(text: str, max_tokens: int = 6000) -> str:
    enc = tiktoken.get_encoding("cl100k_base")
    tokens = enc.encode(text)
    if len(tokens) <= max_tokens:
        return text
    return enc.decode(tokens[:max_tokens])


def load_document(source: str) -> str:
    """Load a resume from a .pdf, .docx, or URL and return its text."""
    src = source.strip()
    lower = src.lower()
    if lower.startswith(("http://", "https://")):
        loader = WebBaseLoader(src)
    elif lower.endswith(".pdf"):
        loader = PyPDFLoader(src)
    elif lower.endswith(".docx"):
        loader = Docx2txtLoader(src)
    else:
        raise ValueError(f"Unsupported format: {source}")
    docs = loader.load()
    full_text = "\n\n".join(d.page_content for d in docs)
    return truncate_text(full_text)


print("Helpers ready")

---
## 3. Prompts

In [ ]:
JOB_MATCH_PROMPT = PromptTemplate(
    template="""You are an expert resume reviewer. Match this resume to the job.

**Score**: X/100
**Overall Match**: X%
**Keywords Matched**: • ...
**Missing Keywords**: • ...
**Readability Score**: X/100
**ATS Compatibility Score**: X/100
**2-liner Summary**: ...
**Skill Gap Analysis**: • ...
**Improvement Suggestions**: • ...
**Industry Specific Feedback**: • ...

Job: {job_description}
Resume: {resume_text}

Be honest and structured.
""",
    input_variables=["resume_text", "job_description"],
)

STANDALONE_PROMPT = PromptTemplate(
    template="""You are an expert resume reviewer. Score this resume on its own merit.

1. **Score**: X/100
2. **Strengths**: • ...
3. **Weaknesses**: • ...
4. **Skills Mentioned**: • ...
5. **Recommended Skills to Add**: • ...
6. **Next Career Steps**: • ...
7. **ATS Compatibility**: X/100
8. **Overall Verdict**: ...

Resume: {resume_text}

Be honest and use the exact structure above.
""",
    input_variables=["resume_text"],
)

print("Prompts ready")

---
## 4. RunnableLambda

`RunnableLambda` wraps **any Python function** so it fits into a `|` chain.

**When to use it:**  
You need custom logic (load a file, clean text, parse a dict) somewhere in the middle of a chain.

In [ ]:
# Wrap load_document so it becomes a Runnable that can sit inside a chain
load_doc_runnable = RunnableLambda(load_document)

# Simple chain: load file → summarise in one sentence
quick_summary_chain = (
    load_doc_runnable
    | RunnableLambda(lambda text: f"Summarise this resume in one sentence:\n{text}")
    | llm
    | parser
)

# Uncomment to test:
# print(quick_summary_chain.invoke("Mamatha Resume word am.docx"))
print("RunnableLambda chain ready")

---
## 5. RunnablePassthrough

`RunnablePassthrough` passes the input to the next step **unchanged**.

**When to use it:**  
You load the resume text from a file, but you also need to carry the `job_description` string  
through to the prompt — it shouldn't be transformed, just forwarded.

In [ ]:
# Input shape: {"source": "resume.docx", "job_description": "Data Analyst"}
# We need to load the file AND keep job_description intact

passthrough_chain = (
    RunnableParallel({
        # Load the file text
        "resume_text": RunnableLambda(lambda x: load_document(x["source"])),
        # Pass job_description through unchanged
        "job_description": RunnablePassthrough() | RunnableLambda(lambda x: x["job_description"]),
    })
    | JOB_MATCH_PROMPT
    | llm
    | parser
)

# Usage:
# result = passthrough_chain.invoke({
#     "source": "Mamatha Resume word am.docx",
#     "job_description": "Data Analyst"
# })
# print(result)
print("RunnablePassthrough chain ready")

---
## 6. RunnableParallel — Dual Review

`RunnableParallel` runs **multiple runnables simultaneously** on the same input and merges  
results into a dict.

**When to use it:**  
Run both a job-match review AND a standalone review in one call, then show both side-by-side.

In [ ]:
# Both branches receive the same dict: {"resume_text": ..., "job_description": ...}
# STANDALONE_PROMPT only uses resume_text (extra keys are ignored)
dual_review_chain = RunnableParallel({
    "job_match_review": JOB_MATCH_PROMPT | llm | parser,
    "standalone_review": STANDALONE_PROMPT | llm | parser,
})

# Usage:
# resume_text = load_document("Mamatha Resume word am.docx")
# result = dual_review_chain.invoke({
#     "resume_text": resume_text,
#     "job_description": "Data Analyst"
# })
# print("=== Job Match ===")
# print(result["job_match_review"])
# print("\n=== Standalone ===")
# print(result["standalone_review"])
print("RunnableParallel dual-review chain ready")

---
## 7. RunnableBranch — Conditional Routing

`RunnableBranch` routes input to **different chains** based on a condition — like an if/elif/else  
at the chain level.

**When to use it:**  
If the user provides a job description → use `JOB_MATCH_PROMPT`.  
If no job description → fall back to `STANDALONE_PROMPT`.

In [ ]:
# Input shape: {"resume_text": str, "job_description": str | ""}

smart_review_chain = RunnableBranch(
    # Branch 1: job_description is provided — use job-match prompt
    (
        lambda x: bool(x.get("job_description", "").strip()),
        JOB_MATCH_PROMPT | llm | parser,
    ),
    # Default branch: no job description — use standalone prompt
    STANDALONE_PROMPT | llm | parser,
)

# Usage:
# resume_text = load_document("Mamatha Resume word am.docx")
#
# With job description:
# result = smart_review_chain.invoke({
#     "resume_text": resume_text,
#     "job_description": "Data Analyst at TCS"
# })
#
# Without job description:
# result = smart_review_chain.invoke({
#     "resume_text": resume_text,
#     "job_description": ""
# })
print("RunnableBranch smart-review chain ready")

---
## 8. Full Auto-Routing Pipeline

Combine the document loader + branching review into a single end-to-end pipeline.

In [ ]:
# Input shape: {"source": "path/to/resume.docx", "job_description": "..."}

full_pipeline = (
    RunnableParallel({
        "resume_text": RunnableLambda(lambda x: load_document(x["source"])),
        "job_description": RunnablePassthrough() | RunnableLambda(lambda x: x.get("job_description", "")),
    })
    | smart_review_chain  # RunnableBranch picks the right prompt
)

# Usage:
# result = full_pipeline.invoke({
#     "source": "Mamatha Resume word am.docx",
#     "job_description": "Frontend Developer"   # leave empty for standalone
# })
# print(result)
print("Full auto-routing pipeline ready")

---
## 9. Memory — Why It Matters

Every chain call above is **stateless** — it forgets the previous conversation.  
With memory the chain can answer follow-up questions like:
- *"What was the first weakness you mentioned?"*
- *"How can I fix it?"*

### Two memory approaches

| Approach | Class | Use for |
|---|---|---|
| Per-session history | `InMemoryChatMessageHistory` + `RunnableWithMessageHistory` | Adding memory to a plain **chain** |
| Agent checkpointing | `MemorySaver` + `create_agent` | Adding memory to a LangGraph **agent** |

---
## 10. Memory Storage — `InMemoryChatMessageHistory`

`InMemoryChatMessageHistory` stores conversation turns **in a Python dict** (lives as long as  
the process). Each user session gets its own `session_id` → its own isolated history.

In [ ]:
# Global dict: session_id → InMemoryChatMessageHistory
session_store: dict = {}


def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """Return (or create) the message history for this session."""
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]


print("Memory store ready")
print("Active sessions:", list(session_store.keys()))

---
## 11. `RunnableWithMessageHistory` — Chain Memory

This wrapper:
1. Loads previous messages for the `session_id` before every call
2. Appends the new human + AI messages after each call

The prompt needs a `MessagesPlaceholder` slot where the history is injected.

In [ ]:
# Chat-style prompt with a history slot
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful resume reviewer assistant."),
    MessagesPlaceholder(variable_name="history"),  # past turns injected here
    ("human", "{input}"),
])

base_chain = chat_prompt | llm | parser

chain_with_memory = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",     # key that holds the user's message
    history_messages_key="history", # key in the prompt for past turns
)

print("Chain with memory created!")

In [ ]:
config = {"configurable": {"session_id": "user_yamuna"}}

# Turn 1 — introduce yourself
r1 = chain_with_memory.invoke(
    {"input": "My name is Yamuna and I am a data analyst."},
    config=config,
)
print("Turn 1:", r1)

# Turn 2 — chain remembers your name and role
r2 = chain_with_memory.invoke(
    {"input": "What are 3 skills I should add to my resume?"},
    config=config,
)
print("\nTurn 2:", r2)

# Inspect the stored history
history = get_session_history("user_yamuna")
print(f"\nMessages in history: {len(history.messages)}")

---
## 12. Resume-Specific Memory Chain

Bake the resume text into the **system prompt** so the model can answer follow-up  
questions about that specific resume across multiple turns.

In [ ]:
# Load resume once — replace with your actual file
# resume_text = load_document("Mamatha Resume word am.docx")

# Placeholder for demo (replace with real load above)
resume_text = "[Paste or load resume text here]"

resume_chat_prompt = ChatPromptTemplate.from_messages([
    ("system", f"""You are an expert resume reviewer.
The user's resume is below. Answer ALL questions based only on this resume.

RESUME:
{resume_text}
"""),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

resume_base_chain = resume_chat_prompt | llm | parser

resume_chain_with_memory = RunnableWithMessageHistory(
    resume_base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

print("Resume memory chain ready!")

In [ ]:
resume_config = {"configurable": {"session_id": "resume_session_yamuna"}}

# Turn 1 — initial review
r1 = resume_chain_with_memory.invoke(
    {"input": "Give me an overall score and the top 3 weaknesses."},
    config=resume_config,
)
print("Turn 1:\n", r1)

# Turn 2 — follow-up (chain remembers the weaknesses from Turn 1)
r2 = resume_chain_with_memory.invoke(
    {"input": "How can I fix the first weakness you mentioned?"},
    config=resume_config,
)
print("\nTurn 2:\n", r2)

---
## 13. Agent Memory — `MemorySaver` Checkpointer

For LangGraph agents created with `create_agent`, memory works via a **checkpointer**.  
It saves the full agent state (messages + tool call results) between turns.

Each conversation thread is identified by `thread_id`.

| Checkpointer | Storage | Persists across restarts? |
|---|---|---|
| `MemorySaver` | In-process dict | No |
| `SqliteSaver` | SQLite file | Yes |

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver


def run_chain(prompt, **kwargs) -> str:
    return (prompt | llm | parser).invoke(kwargs)


@tool
def analyze_resume_job(source: str) -> str:
    """Analyze a resume against a job description. Pass a .pdf/.docx file path or a URL."""
    text = load_document(source)
    return run_chain(JOB_MATCH_PROMPT, resume_text=text, job_description="front desk executive")


@tool
def check_resume_standalone(source: str) -> str:
    """General resume review with no specific job. Pass a .pdf/.docx file path or a URL."""
    text = load_document(source)
    return run_chain(STANDALONE_PROMPT, resume_text=text)


checkpointer = MemorySaver()

agent_with_memory = create_agent(
    model=llm,
    tools=[analyze_resume_job, check_resume_standalone],
    system_prompt="You are a helpful resume reviewer. Remember previous messages in this thread.",
    checkpointer=checkpointer,  # enables per-thread memory
)

print("Agent with MemorySaver created!")

In [ ]:
# thread_id identifies the conversation — same id = agent remembers previous turns
agent_config = {"configurable": {"thread_id": "thread_yamuna_1"}}

# Turn 1 — agent loads and reviews the resume
r1 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "Check my resume at: Mamatha Resume word am.docx"}]},
    config=agent_config,
)
print("Turn 1:\n", r1["messages"][-1].content)

# Turn 2 — agent remembers the resume was already reviewed
r2 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "What 3 skills should I add based on your review?"}]},
    config=agent_config,
)
print("\nTurn 2:\n", r2["messages"][-1].content)

---
## 14. Inspect Stored Memory

Both memory stores are just Python objects — easy to inspect for debugging.

In [ ]:
# ── Chain memory (InMemoryChatMessageHistory) ──────────────────────────────────
print("Active sessions:", list(session_store.keys()))
print()

for sid, history in session_store.items():
    print(f"Session: {sid}  ({len(history.messages)} messages)")
    for msg in history.messages:
        role = msg.__class__.__name__.replace("Message", "")
        preview = msg.content[:80].replace("\n", " ")
        print(f"  [{role}] {preview}...")
    print()

In [ ]:
# ── Agent memory (MemorySaver checkpointer) ────────────────────────────────────
# MemorySaver stores state as checkpoints keyed by thread_id
# Access via the checkpointer's internal store
saved_threads = list(checkpointer.storage.keys()) if hasattr(checkpointer, "storage") else []
print("Agent threads in checkpointer:", saved_threads or "(use checkpointer.get() to inspect)")

---
## Summary

| Concept | Class | When to use |
|---|---|---|
| Wrap a function | `RunnableLambda` | Any custom logic in the middle of a chain |
| Forward unchanged | `RunnablePassthrough` | Carry original data alongside transformed data |
| Run in parallel | `RunnableParallel` | Get two reviews at once, compare outputs |
| Conditional route | `RunnableBranch` | Switch prompt/chain based on input condition |
| Chain memory | `InMemoryChatMessageHistory` + `RunnableWithMessageHistory` | Multi-turn Q&A over a chain |
| Agent memory | `MemorySaver` checkpointer | Multi-turn conversations with a tool-calling agent |

**Key rule:**  
- Use `RunnableWithMessageHistory` when your logic lives in a chain (`prompt | llm | parser`).  
- Use `MemorySaver` (or `SqliteSaver`) when your logic lives in a LangGraph agent.

In [ ]:
# 10th-june - 11- june thursday same timings
# 24th june - 22th june sunday same timings